---
title: "Exercise 10: Genetic Correlation"
subtitle: "Post-GWAS Analysis Course"
format:
  html:
    embed-resources: true
    toc: true
    toc-depth: 3
execute:
  message: false
  warning: false
jupyter: python
---

# Overview

This notebook uses LDSC to estimate pairwise genetic correlations between ADHD and other traits. You will read the correlation table, inspect the forest plot, and compare the patterns across trait pairs.

::: {.callout-note}
## Learning goals
By the end of this notebook, you should be able to:
- explain what a genetic correlation measures
- run pairwise LDSC correlation analyses from munged summary statistics
- interpret the result table and forest plot for evidence of shared genetic architecture
:::

::: callout-warning
You will need the following input files: GWAS sumstats for ADHD, BMI, educational attainment, age at first birth, and depression. Those are available as input data.
:::


 ## Table of Contents

* [Set up](#section_1)     
* [Run LDSC to estimate pairwise genetic correlations](#section_2)    
* [Analyse and present results](#section_3)
  * [Result table](#section_3_1)
  * [Forest plot](#section_3_2)

# 1. Set up <a class="anchor" id="section_1"></a>

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

## 2. Run LDSC to estimate pairwise genetic correlations <a class="anchor" id="section_2"></a>

During the previous workshop, we have harmonized the sumstats using the munge function, we do not need to repeat that step. We will now use the munged files to calculate genetic correlations between pairs of traits with LDSC. The function we use is very similar to the one we used to calculate SNP-based h2.
We will calculate the genetic correlation between ADHD and all the other traits.

In [ ]:
#Example of command for ADHD and BMI

command = "ldsc \
--rg input/ldsc/adhd_pgc_2022_munged.sumstats.gz,input/ldsc/bmi_yengo_2018_munged.sumstats.gz \
--ref-ld-chr reference_data/eur_w_ld_chr/ \
--w-ld-chr reference_data/eur_w_ld_chr/ \
--n-blocks 200 \
--out output/ldsc/adhd_bmi_rg" 

os.system(command)

In [ ]:
#You can now adapt this command to calculate the rg between ADHD and the other traits

#ADHD and educational attainment

command = "ldsc \
--rg input/ldsc/___,input/ldsc/___ \
--ref-ld-chr reference_data/eur_w_ld_chr/ \
--w-ld-chr reference_data/eur_w_ld_chr/ \
--n-blocks 200 \
--out output/ldsc/adhd_edu_rg" 

os.system(command)


In [ ]:
#ADHD and Year at first birth

command = "ldsc \
--rg input/ldsc/___,input/ldsc/___ \
--ref-ld-chr reference_data/eur_w_ld_chr/ \
--w-ld-chr reference_data/eur_w_ld_chr/ \
--n-blocks 200 \
--out output/ldsc/adhd_yearfb_rg" 

os.system(command)


In [ ]:
#ADHD and Depression

command = "ldsc \
--rg input/ldsc/___,input/ldsc/___ \
--ref-ld-chr reference_data/eur_w_ld_chr/ \
--w-ld-chr reference_data/eur_w_ld_chr/ \
--n-blocks 200 \
--out output/ldsc/adhd_depression_rg" 

os.system(command)


## 3. Analyse and present results <a class="anchor" id="section_3"></a>

You now have calculated the SNP-based heritability for each of the trait. We will extract that information from the log file, and present the results.

### 3.1 Result table <a class="anchor" id="section_3_1"></a>
If you open the log files generated in the previous section, you can see that they contain a lot of information, including the genetic correlation (rg) that we will extract for each pair of traits and present in a table.

In [ ]:
#Create a list with the file paths ----> Adapt this section based on how you named the files
path_list = ["output/ldsc/adhd_bmi_rg.log", 
             "output/ldsc/adhd_edu_rg.log", 
             "output/ldsc/adhd_yearfb_rg.log", 
             "output/ldsc/adhd_depression_rg.log"]

#Create a list of trait names
trait_list = ["bmi", "educational_attainment", "year_first_birth", "depression"]

#Initiate an empty list 
rows = []

#We iterate over all the files 
for i in range(len(path_list)):
    
    #Initiate the result vector
    data = {
        "trait1": "adhd",
        "trait2": trait_list[i],
        "rg": None,
        "se": None,
        "p-value": None
    }

    # Read the file and extract rg, se and p-value
    with open(path_list[i], "r") as f:
        for j, line in enumerate(f):
            if j == 54:
                words = line.strip().split()
                
                # Extract 3th and 4th "words" (indices 2 and 3) -> corresponds to rg and se
                data["rg"] = float(words[2]) 
                data["se"] = float(words[3].strip("()"))

            elif j == 56:
                words = line.strip().split()

                #Extract the 2nd "word" (indice 1) -> corresponds to the p-value
                data["p-value"] = float(words[1])

                break # no need to continue reading the file

        #Store in rows
        rows.append(data)
    
   
        
# Store in a DataFrame
df = pd.DataFrame(rows)

df

### 3.2 Present results in a figure <a class="anchor" id="section_3_2"></a>

From the table we just created, we can now create a figure to represent the results. A common way of representing rg is to use a forest plot.

In [ ]:
#Create plot 
fig,ax = plt.subplots(figsize=(6,4))

#Error bars -> SE
ax.errorbar(
    df["rg"], 
    df["trait2"], 
    xerr=[abs(df["rg"]-df["se"]), abs(df["rg"]-df["se"])], 
    fmt='o', 
    color='black', 
)

#Vertical line at 0
ax.axvline(0, color='black', linestyle='--')

#Labels on the y-axis
ax.set_yticklabels(df['trait2'])

#Labels on x-axis
ax.set_xlabel("Genetic correlation")

#Add p-value on the right
for i, (_, row) in enumerate(df.iterrows()):
    ax.text(
        row["rg"] + row["se"] + 0.03,
        i,
        f"p={row['p-value']:.2e}",
        va='bottom',
        fontsize=8
    )

plt.tight_layout()
plt.show()

::: callout-note
Based on the result table and the figure, answer the following questions

- Are the results in line with what you expect/the literature?

- Do we need to consider multiple testing? What method can be applied to correct for it? Does it impact the results here?
:::